# Построение векторной базы из markdown

Ноутбук читает `*.md` из `data/`, режет текст на чанки `800/200`,
строит FAISS-индекс на `bge-m3` и сохраняет совместимый формат в `vector_db/`.

In [ ]:
from pathlib import Path
import json

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
DATA_DIR = Path("data")
VECTOR_DB_DIR = Path("vector_db")
EMBEDDING_MODEL_PATH = "/home/vladislav/models/bge-m3"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200
EMBEDDING_DEVICE = "cpu"  # Для стабильного локального запуска. Если GPU совместим, поставьте "cuda"

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("CHUNK_OVERLAP должен быть меньше CHUNK_SIZE")

if EMBEDDING_DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("EMBEDDING_DEVICE='cuda', но CUDA недоступна")

def read_markdown_documents(data_dir: Path) -> list[dict]:
    docs = []
    for path in sorted(data_dir.glob("*.md")):
        text = path.read_text(encoding="utf-8").strip()
        if text:
            docs.append({"source": str(path), "text": text})
    return docs

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    step = chunk_size - overlap
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start += step
    return chunks

def build_chunks(docs: list[dict]) -> list[dict]:
    rows = []
    for doc in docs:
        pieces = chunk_text(doc["text"])
        for i, piece in enumerate(pieces):
            rows.append({
                "text": piece,
                "source": doc["source"],
                "chunk_id": i,
            })
    return rows

In [ ]:
docs = read_markdown_documents(DATA_DIR)
if not docs:
    raise FileNotFoundError(f"Не найдено markdown-документов в {DATA_DIR.resolve()}")

chunks = build_chunks(docs)
texts = [item["text"] for item in chunks]

print(f"Используется устройство для эмбеддингов: {EMBEDDING_DEVICE}")
embedder = SentenceTransformer(EMBEDDING_MODEL_PATH, device=EMBEDDING_DEVICE)
vectors = embedder.encode(
    texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
    device=EMBEDDING_DEVICE,
).astype(np.float32)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(VECTOR_DB_DIR / "index.faiss"))

docs_payload = [
    {
        "text": item["text"],
        "source": item["source"],
        "chunk_id": item["chunk_id"],
    }
    for item in chunks
]
(VECTOR_DB_DIR / "docs.json").write_text(
    json.dumps(docs_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Документов: {len(docs)}")
print(f"Чанков: {len(chunks)}")
print(f"Размерность эмбеддинга: {vectors.shape[1]}")
print(f"Сохранено: {(VECTOR_DB_DIR / 'index.faiss').resolve()}")
print(f"Сохранено: {(VECTOR_DB_DIR / 'docs.json').resolve()}")

In [ ]:
# Мини-проверка поиска по построенному индексу
query = "Как в городе улучшили транспортную связность районов?"
query_vec = embedder.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True,
    device=EMBEDDING_DEVICE,
).astype(np.float32)
scores, idxs = index.search(query_vec, k=3)

for rank, (score, idx) in enumerate(zip(scores[0], idxs[0]), start=1):
    if idx < 0:
        continue
    chunk = docs_payload[idx]
    preview = chunk["text"][:220].replace("\n", " ")
    print(f"{rank}. score={score:.4f} | source={chunk['source']} | chunk_id={chunk['chunk_id']}")
    print(preview)
    print('-' * 80)